# SpaceHub Calc Optimization Verification

This notebook compares the output values of three versions of `spacehub_calc.py`:
1. **spacehub_calc.py** - Fully vectorized (current)
2. **spacehub_calc_notoptimized.py** - Original with no optimizations
3. **spacehub_calc_notvectorized.py** - Quick fix only (no vectorization)

All orbital element values should match across implementations.

In [1]:
import numpy as np
import time

# Import all three versions
import spacehub_calc as sc_vec  # Fully vectorized
import spacehub_calc_notoptimized as sc_orig  # Original
import spacehub_calc_notvectorized as sc_quick  # Quick fix only

In [2]:
# Test data file - adjust path as needed
test_file = "KeplerSimpleNoZeros.dat"  # Change to your test file

In [3]:
# Create TwoBodyOrbit objects from each implementation
print("Creating TwoBodyOrbit objects...")

t0 = time.time()
orb_vec = sc_vec.TwoBodyOrbit(test_file)
t_vec = time.time() - t0
print(f"Vectorized version: {t_vec:.4f}s")

t0 = time.time()
orb_orig = sc_orig.TwoBodyOrbit(test_file)
t_orig = time.time() - t0
print(f"Original version: {t_orig:.4f}s")

t0 = time.time()
orb_quick = sc_quick.TwoBodyOrbit(test_file)
t_quick = time.time() - t0
print(f"Quick-fix version: {t_quick:.4f}s")

print(f"\nSpeedup vs original: {t_orig/t_vec:.1f}x")
print(f"Speedup vs quick-fix: {t_quick/t_vec:.1f}x")

Creating TwoBodyOrbit objects...
load data complete
Determining timesteps...
Calculating orbital state vectors...
Calculating scalar orbital elements...
Calculated c0 values are within 0.04638 of constant (maximum - minimum)
Done
Vectorized version: 0.7232s
load data complete
Determining timesteps...
Calculating orbital state vectors...
Calculating scalar orbital elements...
Calculated c0 values are within 0.04638 of constant (maximum - minimum)
Done
Original version: 63.8782s
load data complete
Determining timesteps...
Calculating orbital state vectors...
Calculating scalar orbital elements...
Calculated c0 values are within 0.04638 of constant (maximum - minimum)
Done
Quick-fix version: 23.0047s

Speedup vs original: 88.3x
Speedup vs quick-fix: 31.8x


## Compare Orbital Elements

Define comparison function and tolerance.

In [4]:
def compare_arrays(arr1, arr2, name, tol=1e-10):
    """
    Compare two arrays and report maximum difference.
    Returns True if arrays match within tolerance.
    """
    a1 = np.asarray(arr1)
    a2 = np.asarray(arr2)
    
    if a1.shape != a2.shape:
        print(f"  {name}: SHAPE MISMATCH - {a1.shape} vs {a2.shape}")
        return False
    
    # Handle NaN values
    nan_mask = np.isnan(a1) | np.isnan(a2)
    if np.any(nan_mask):
        # Check if NaNs are in the same positions
        if not np.array_equal(np.isnan(a1), np.isnan(a2)):
            print(f"  {name}: NaN POSITION MISMATCH")
            return False
        # Compare non-NaN values only
        a1_valid = a1[~nan_mask]
        a2_valid = a2[~nan_mask]
        if len(a1_valid) == 0:
            max_diff = 0
        else:
            max_diff = np.max(np.abs(a1_valid - a2_valid))
    else:
        max_diff = np.max(np.abs(a1 - a2))
    
    passed = max_diff <= tol
    status = "PASS" if passed else "FAIL"
    print(f"  {name}: max_diff = {max_diff:.2e} [{status}]")
    return passed

In [5]:
# List of orbital elements to compare
elements = [
    ('semiMajorAxis', 'Semi-major axis'),
    ('eccentricity', 'Eccentricity'),
    ('inclination_rad', 'Inclination (rad)'),
    ('inclination_deg', 'Inclination (deg)'),
    ('LongitudeAscendingNode', 'Long. Asc. Node (rad)'),
    ('LongitudeAscendingNode_deg', 'Long. Asc. Node (deg)'),
    ('argument_of_periapsis_rad', 'Arg. Periapsis (rad)'),
    ('argument_of_periapsis_deg', 'Arg. Periapsis (deg)'),
    ('true_anomaly_rad', 'True anomaly (rad)'),
    ('true_anomaly_deg', 'True anomaly (deg)'),
    ('eccentric_anomaly_rad', 'Eccentric anomaly (rad)'),
    ('eccentric_anomaly_deg', 'Eccentric anomaly (deg)'),
    ('time_of_pericenter_passage', 'Time of pericenter'),
    ('orbital_period', 'Orbital period'),
    ('c0', 'Peters c0'),
    ('sinOmega', 'sin(Omega)'),
    ('cosOmega', 'cos(Omega)'),
    ('sinf', 'sin(f)'),
    ('cosf', 'cos(f)'),
]

In [6]:
# Compare vectorized vs original
print("=" * 60)
print("COMPARISON: Vectorized vs Original")
print("=" * 60)

tol = 1e-10
all_passed_vo = True

for attr, name in elements:
    try:
        arr_vec = getattr(orb_vec, attr)
        arr_orig = getattr(orb_orig, attr)
        passed = compare_arrays(arr_vec, arr_orig, name, tol)
        all_passed_vo = all_passed_vo and passed
    except AttributeError as e:
        print(f"  {name}: ATTRIBUTE MISSING - {e}")
        all_passed_vo = False

print()
if all_passed_vo:
    print("ALL TESTS PASSED: Vectorized matches Original")
else:
    print("SOME TESTS FAILED: Check differences above")

COMPARISON: Vectorized vs Original
  Semi-major axis: max_diff = 4.16e-17 [PASS]
  Eccentricity: max_diff = 3.21e-15 [PASS]
  Inclination (rad): max_diff = 4.44e-16 [PASS]
  Inclination (deg): max_diff = 2.84e-14 [PASS]
  Long. Asc. Node (rad): max_diff = 1.11e-15 [PASS]
  Long. Asc. Node (deg): max_diff = 6.39e-14 [PASS]
  Arg. Periapsis (rad): max_diff = 1.71e-14 [PASS]
  Arg. Periapsis (deg): max_diff = 9.81e-13 [PASS]
  True anomaly (rad): max_diff = 1.08e-12 [PASS]
  True anomaly (deg): max_diff = 6.21e-11 [PASS]
  Eccentric anomaly (rad): max_diff = 3.03e-13 [PASS]
  Eccentric anomaly (deg): max_diff = 1.74e-11 [PASS]
  Time of pericenter: max_diff = 5.68e-14 [PASS]
  Orbital period: max_diff = 1.39e-16 [PASS]
  Peters c0: max_diff = 7.66e-15 [PASS]
  sin(Omega): max_diff = 7.77e-16 [PASS]
  cos(Omega): max_diff = 7.77e-16 [PASS]
  sin(f): max_diff = 1.08e-12 [PASS]
  cos(f): max_diff = 2.64e-15 [PASS]

ALL TESTS PASSED: Vectorized matches Original


In [7]:
# Compare vectorized vs quick-fix
print("=" * 60)
print("COMPARISON: Vectorized vs Quick-Fix")
print("=" * 60)

all_passed_vq = True

for attr, name in elements:
    try:
        arr_vec = getattr(orb_vec, attr)
        arr_quick = getattr(orb_quick, attr)
        passed = compare_arrays(arr_vec, arr_quick, name, tol)
        all_passed_vq = all_passed_vq and passed
    except AttributeError as e:
        print(f"  {name}: ATTRIBUTE MISSING - {e}")
        all_passed_vq = False

print()
if all_passed_vq:
    print("ALL TESTS PASSED: Vectorized matches Quick-Fix")
else:
    print("SOME TESTS FAILED: Check differences above")

COMPARISON: Vectorized vs Quick-Fix
  Semi-major axis: max_diff = 4.16e-17 [PASS]
  Eccentricity: max_diff = 3.21e-15 [PASS]
  Inclination (rad): max_diff = 4.44e-16 [PASS]
  Inclination (deg): max_diff = 2.84e-14 [PASS]
  Long. Asc. Node (rad): max_diff = 1.11e-15 [PASS]
  Long. Asc. Node (deg): max_diff = 6.39e-14 [PASS]
  Arg. Periapsis (rad): max_diff = 1.71e-14 [PASS]
  Arg. Periapsis (deg): max_diff = 9.81e-13 [PASS]
  True anomaly (rad): max_diff = 1.08e-12 [PASS]


  True anomaly (deg): max_diff = 6.21e-11 [PASS]
  Eccentric anomaly (rad): max_diff = 3.03e-13 [PASS]
  Eccentric anomaly (deg): max_diff = 1.74e-11 [PASS]
  Time of pericenter: max_diff = 5.68e-14 [PASS]
  Orbital period: max_diff = 1.39e-16 [PASS]
  Peters c0: max_diff = 7.66e-15 [PASS]
  sin(Omega): max_diff = 7.77e-16 [PASS]
  cos(Omega): max_diff = 7.77e-16 [PASS]
  sin(f): max_diff = 1.08e-12 [PASS]
  cos(f): max_diff = 2.64e-15 [PASS]

ALL TESTS PASSED: Vectorized matches Quick-Fix


In [8]:
# Compare quick-fix vs original (sanity check)
print("=" * 60)
print("COMPARISON: Quick-Fix vs Original (sanity check)")
print("=" * 60)

all_passed_qo = True

for attr, name in elements:
    try:
        arr_quick = getattr(orb_quick, attr)
        arr_orig = getattr(orb_orig, attr)
        passed = compare_arrays(arr_quick, arr_orig, name, tol)
        all_passed_qo = all_passed_qo and passed
    except AttributeError as e:
        print(f"  {name}: ATTRIBUTE MISSING - {e}")
        all_passed_qo = False

print()
if all_passed_qo:
    print("ALL TESTS PASSED: Quick-Fix matches Original")
else:
    print("SOME TESTS FAILED: Check differences above")

COMPARISON: Quick-Fix vs Original (sanity check)
  Semi-major axis: max_diff = 0.00e+00 [PASS]
  Eccentricity: max_diff = 0.00e+00 [PASS]
  Inclination (rad): max_diff = 0.00e+00 [PASS]
  Inclination (deg): max_diff = 0.00e+00 [PASS]
  Long. Asc. Node (rad): max_diff = 0.00e+00 [PASS]
  Long. Asc. Node (deg): max_diff = 0.00e+00 [PASS]
  Arg. Periapsis (rad): max_diff = 0.00e+00 [PASS]
  Arg. Periapsis (deg): max_diff = 0.00e+00 [PASS]
  True anomaly (rad): max_diff = 0.00e+00 [PASS]
  True anomaly (deg): max_diff = 0.00e+00 [PASS]
  Eccentric anomaly (rad): max_diff = 0.00e+00 [PASS]
  Eccentric anomaly (deg): max_diff = 0.00e+00 [PASS]
  Time of pericenter: max_diff = 0.00e+00 [PASS]
  Orbital period: max_diff = 0.00e+00 [PASS]
  Peters c0: max_diff = 0.00e+00 [PASS]
  sin(Omega): max_diff = 0.00e+00 [PASS]
  cos(Omega): max_diff = 0.00e+00 [PASS]
  sin(f): max_diff = 0.00e+00 [PASS]
  cos(f): max_diff = 0.00e+00 [PASS]

ALL TESTS PASSED: Quick-Fix matches Original


## Summary

In [9]:
print("=" * 60)
print("FINAL SUMMARY")
print("=" * 60)
print(f"\nTiming:")
print(f"  Vectorized: {t_vec:.4f}s")
print(f"  Quick-fix:  {t_quick:.4f}s")
print(f"  Original:   {t_orig:.4f}s")
print(f"\nSpeedups:")
print(f"  Vectorized vs Original:   {t_orig/t_vec:.1f}x faster")
print(f"  Vectorized vs Quick-fix:  {t_quick/t_vec:.1f}x faster")
print(f"\nAccuracy (tol={tol}):")
print(f"  Vectorized vs Original:   {'PASS' if all_passed_vo else 'FAIL'}")
print(f"  Vectorized vs Quick-fix:  {'PASS' if all_passed_vq else 'FAIL'}")
print(f"  Quick-fix vs Original:    {'PASS' if all_passed_qo else 'FAIL'}")

FINAL SUMMARY

Timing:
  Vectorized: 0.7232s
  Quick-fix:  23.0047s
  Original:   63.8782s

Speedups:
  Vectorized vs Original:   88.3x faster
  Vectorized vs Quick-fix:  31.8x faster

Accuracy (tol=1e-10):
  Vectorized vs Original:   PASS
  Vectorized vs Quick-fix:  PASS
  Quick-fix vs Original:    PASS
